In [1]:
import pandas as pd
import sys
import os
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import Image
#import plotly.io as pio
#pio.renderers.default = "vscode" #   notebook      iframe

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import plot_histogram, plot_univariate_freq
from utils import g, describe_dataset, format_number
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = '#333333'> ANÁLISIS UNIVARIADO

## <font color = 'gray'> OBJETIVO

Analizar el comportamiento individual de las variables y detectar errores en los datos.

## <font color = 'gray'> METODOLOGÍA

El análisis univariado se realiza de manera independiente para cada variable con el objetivo de comprender su distribución y comportamiento dentro del conjunto de datos. 

Se diferencian dos tipos de variables en el análisis, aplicando metodologías específicas para cada caso.

* Variables Categóricas:

    - Se analiza la distribución de frecuencias de cada categoría dentro de la variable.
    - Se calcula el peso relativo o proporción de cada categoría en relación con el total de observaciones.
    - Se utilizan representaciones gráficas, como diagramas de barras, para visualizar la composición de cada variable.

* Variables Continuas:

    - Se estudia la distribución de los valores mediante histogramas y gráficos de densidad.
    - Se calculan medidas estadísticas clave, como la media, mediana, desviación estándar, mínimo y máximo.
    - Se analizan posibles asimetrías en la distribución y la presencia de valores atípicos.

# <font color = '#333333'> GENERALIDADES


In [2]:
df = pd.read_csv(get_data_path("Credit_score_cleaned_data.csv"))

In [3]:
res = check_dataframe_quality(df, verbose = False)

In [4]:
num_rows, num_cols = df.shape

s = {'Fuente de los datos':'https://www.kaggle.com/code/ayushsharma0812/credit-score-classification-part-1-data-cleaning/notebook',
     'Número de filas': format_number(num_rows),
     'Número de variables': format_number(num_cols - 1),
     'Categorías de la variable dependiente': ', '.join(df['Credit_Score'].unique()),
     'Número de clientes únicos': format_number(len(df['Customer_ID'].unique())),
     'Valores faltantes':res['Missing values'],
     'Valores infinitos':res['Infinite values'],
     'Filas Duplicadas':res['Duplicate rows']}

describe_dataset(s, separator_length=125)

-----------------------------------------------------------------------------------------------------------------------------
Fuente de los datos: https://www.kaggle.com/code/ayushsharma0812/credit-score-classification-part-1-data-cleaning/notebook
-----------------------------------------------------------------------------------------------------------------------------
Número de filas: 100,000
-----------------------------------------------------------------------------------------------------------------------------
Número de variables: 31
-----------------------------------------------------------------------------------------------------------------------------
Categorías de la variable dependiente: Standard, Good, Poor
-----------------------------------------------------------------------------------------------------------------------------
Número de clientes únicos: 12,500
--------------------------------------------------------------------------------------------------------

## <font color = 'darkblue'> AGE


Las edades van desde los 14 a los 56 años.

La distribución de las edades es relativamente simétrica: la media y la mediana son iguales, prácticamente.

Es raro deudores con tan poca edad.

In [5]:
fig = plot_histogram(df = df, column='Age', figsize = (800, 500))
fig.show()

In [6]:
g(df[['Age']].describe(percentiles=[0.15, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).transpose(), 0)

,count,mean,std,min,15%,25%,50%,75%,90%,95%,99%,max
Age,"100,000",33,11,14,21,24,33,42,48,52,55,56


# <font color = 'darkblue'> OCCUPATION

La ocupación es una variable categórica con valores como 'Teacher', 'Engineer', 'Doctor', 'Lawyer', entre otros. 

Todas las categorías de esta variable pesan más que el 5%.

In [7]:
fig = plot_univariate_freq(df, 'Occupation', figsize=(900, 500), title=None, normalize=True)
fig.show()

# <font color = 'darkblue'> ANNUAL_INCOME

La distribución de los ingresos anuales tiene una forma típica: 

presenta una asimetría positiva (sesgo a la derecha), donde la mediana es menor que el promedio. 

Esto indica que hay menos casos de deudores con altos ingresos en comparación con aquellos que tienen ingresos más bajos.

Los ingresos anuales oscilan desde los 7 mil a los 180 mil aproximadamente.

La mediana de los ingresos anuales es de 37 mil.

In [8]:
fig = plot_histogram(df, 'Annual_Income', figsize=(900, 500))
fig.show()

In [9]:
g(df[['Annual_Income']].describe(percentiles=[0.15, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).transpose(), 0)

,count,mean,std,min,15%,25%,50%,75%,90%,95%,99%,max
Annual_Income,"100,000","50,505","38,299","7,006","15,950","19,343","37,000","71,683","108,786","130,290","166,892","179,987"


# <font color = 'darkblue'> MONTHLY_INHAND_SALARY

El salario mensual en mano va desde los 304 a los 15,000 dólares. 

La distribución del salario mensual es asimétrica, con una mayor concentración en los valores más bajos.

In [10]:
plot_histogram(df, 'Monthly_Inhand_Salary', figsize=(900, 500))

# <font color = 'darkblue'> NUM_BANK_ACCOUNTS

El número de cuentas bancarias va desde 0 a 10. 

La distribución del número de cuentas bancarias es relativamente simétrica, con una media alrededor de 5.

In [11]:
plot_univariate_freq(df, 'Num_Bank_Accounts', figsize=(900, 500), normalize=True)

In [12]:
g(df[['Num_Bank_Accounts']].describe(percentiles=[0.15, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).transpose(), 0)

,count,mean,std,min,15%,25%,50%,75%,90%,95%,99%,max
Num_Bank_Accounts,"100,000",5,3,0,3,3,5,7,9,10,10,11


# <font color = 'darkblue'> NUM_CREDIT_CARD

El número de tarjetas de crédito va desde 0 a 10. 

La distribución del número de tarjetas de crédito es relativamente simétrica, con una media alrededor de 6.

In [13]:
plot_univariate_freq(df, 'Num_Credit_Card', figsize=(900, 500), normalize=True)

# <font color = 'darkblue'> INTEREST_RATE

Las tasas de interés van desde el 1% al 34%. 

La distribución de la tasa de interés es relativamente simétrica, pero hay tasas de interés muy bajas, lo cual es raro.

In [ ]:
fig = plot_histogram(df, 'Interest_Rate', figsize=(900, 500))
fig.show(renderer = 'vscode')

In [ ]:
g(df[['Interest_Rate']].describe(percentiles=[0.15, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).transpose(), 0)

,count,mean,std,min,15%,25%,50%,75%,90%,95%,99%,max
Interest_Rate,"100,000",15,9,1,5,7,13,20,28,31,34,34


# <font color = 'darkblue'> NUM_OF_LOAN

El número de préstamos va desde 0 a 9. La distribución del número de préstamos es relativamente simétrica, con una media de 4.

# <font color = 'darkblue'> DELAY_FROM_DUE_DATE

El retraso desde la fecha de vencimiento va desde -5 a 76 días. El 90% de los casos llega a lo sumo a 45.

# <font color = 'darkblue'> NUM_OF_DELAYED_PAYMENT

El número de pagos retrasados oscila entre 0 y 28 pagos. Con una media alrededor de 13.

# <font color = 'darkblue'> CHANGED_CREDIT_LIMIT

# <font color = 'red'> Pendiente revisar - El cambio en el límite de crédito entre 0 y 28%, con una media de 10.39.

# <font color = '#333333'> CREDIT_MIX

La mezcla de crédito es una variable categórica con valores como 'Good', 'Average', 'Bad'. Los malos son 23.77%.

# <font color = '#333333'> OUTSTANDING_DEBT

La deuda pendiente va desde 0.23 a casi 5000 dólares. La distribución del Saldo pendiente tiene una asimetría positiva

# <font color = '#333333'> CREDIT_UTILIZATION_RATIO

La tasa de utilización del crédito oscila entre 20% y 50%. Es relativamente simétrica: la utilización promedio y mediana es de alrededor del 32%.

# <font color = '#333333'> CREDIT_HISTORY_AGE

La antiguedad del cliente como deudor oscila de 1 a 404 meses. El promedio es de 221 meses (18 años aprox.)

# <font color = '#333333'> PAYMENT_OF_MIN_AMOUNT

El pago del monto mínimo es una variable categórica con valores como 'Yes', 'No'. Aproximadamente el 40% de los clientes no paga el monto mínimo.

# <font color = '#333333'> TOTAL_EMI_PER_MONTH

Los montos de las cuotas oscilan de -1 a 357 dólares. El promedio de USD 88

# <font color = '#333333'> AMOUNT_INVESTED_MONTHLY

Oscila de 0 a 2.000 USD. El monto del cliente invertido mensualmente se distribuye con una fuerte asimetría positiva, la mayoría invierte de de 50 a 150 USD.

# <font color = '#333333'> PAYMENT_BEHAVIOUR

# <font color = 'red'> Pendiente revisar - El comportamiento de pago es una variable categórica. La categoría más riesgosa debería ser la de alto gasto con pagos pequeños, la cual pesa un 20.81%.

# <font color = '#333333'> MONTHLY_BALANCE

El balance mensual va desde 0 a 1,600 dólares. La distribución del balance mensual es asimétrica, con una mayor concentración en los valores más bajos.

# <font color = '#333333'> LAST_LOAN

El último préstamo es una variable categórica sobre el tipo del último préstamo. Quienes no tienen préstamos representan 5%. Algunos tipos de préstamo pueden ser más riesgosos por naturaleza.